In [2]:
!pip install langchain langchain-community langchain-google-genai

In [ ]:
import os
# google AI studio
os.environ["GOOGLE_API_KEY"]="YOUR-API-KEY"

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0,
    max_tokens = 1000,
    timeout = 30,
    max_retires = 3,

)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: WARNING! max_retires is not default parameter.
                max_retires was transferred to model_kwargs.
                Please confirm that max_retires is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-3.5-fla...': 3}, 'base_url': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [1]:
from langchain_core.tools import tool

@tool
def notify_student(student_id: str, message: str) -> str:
    """
    Send a notification message to a student.
    Use this tool when the user asks to notify, inform, or send a message to a student.
    """
    return str({
        "student_id": student_id,
        "message": message,
        "notification_id": 1,
        "duplicate": False,
        "status": "sent"
    })

In [2]:
@tool
def get_student(student_id: str) -> str:
    """
    Get student information using the student's roll number.
    """
    students = {
        "22CS045": {
            "roll": "22CS045",
            "name": "Priya Raman",
            "branch": "CSE"
        },
        "22IT017": {
            "roll": "22IT017",
            "name": "Arun Kumar",
            "branch": "IT"
        },
        "22ME008": {
            "roll": "22ME008",
            "name": "Karthik Raj",
            "branch": "MECH"
        }
    }

    student = students.get(student_id)

    if not student:
        return '{"error":"unknown_student"}'

    return str(student)

In [3]:
@tool
def book_interview_slot(student_id: str, slot_id: int) -> str:
    """
    Book an interview slot for a student.
    Use this tool when a student wants to book an available interview slot.
    """
    return str({
        "student_id": student_id,
        "slot_id": slot_id,
        "status": "booked",
        "already_booked": False
    })

In [4]:
# Student Placement Agent

student_agent = create_agent(
    model=llm,
    tools=[
        notify_student,
        get_student,
        book_interview_slot
    ],
    system_prompt="""
You are a Student Placement Assistant.

You have exactly three tools:

1. get_student
   - Use this when the user asks for student details.
   - Use the student's roll number.

2. book_interview_slot
   - Use this when the user wants to book an interview slot.
   - Use the student's ID and the requested slot ID.
   - Clearly report whether the booking succeeded or failed.

3. notify_student
   - Use this when the user wants to send a notification to a student.
   - Include the student's ID and the notification message.

Choose the appropriate tool based on the user's request.
Do not invent information.
After using a tool, give the user a clear and concise response.
"""
)

NameError: name 'create_agent' is not defined

In [ ]:
response = student_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Get the details of student 22CS045"
        }
    ]
})

print(response["messages"][-1].content)

In [ ]:
response = student_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Book interview slot 1 for 22CS045"
        }
    ]
})

print(response["messages"][-1].content)

In [ ]:
response = student_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Notify 22CS045 that their interview is confirmed"
        }
    ]
})

print(response["messages"][-1].content)